# Étape 7 — Phase 2 : confirmation de l'effet du graphe (5 graines)

L'étape 6 a montré, avec **une seule graine**, que le graphe expert aide sur les **premières tentatives** quand les données sont rares.
Ce notebook vérifie que cet effet est **réel et stable** avant de lancer la grille complète de RQ2.

Ce qui change par rapport à l'étape 6 :
- **5 graines** : à chaque graine, un autre échantillon d'élèves et un autre graphe aléatoire ;
- **jusqu'à 60 époques** (arrêt anticipé après 5 époques sans progrès), pour que les modèles convergent vraiment ;
- régimes **1 %, 5 % et 10 %** des élèves d'entraînement ;
- **tests statistiques appariés** : pour une graine donnée, les trois graphes sont entraînés sur les **mêmes élèves**.

**Question décisive :** l'écart **expert − aléatoire** sur les premières tentatives est-il positif sur les 5 graines ?

**GPU T4 obligatoire.** Durée : environ 1 h 30 à 2 h. La cellule 7 reprend là où elle s'est arrêtée en cas de déconnexion.

In [ ]:
# Cellule 1 — Drive, GPU, paramètres
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy import stats

DOSSIER   = '/content/drive/MyDrive/projet_kt'
PREP      = os.path.join(DOSSIER, 'junyi_prepare')
RESULTATS = os.path.join(DOSSIER, 'resultats')
os.makedirs(RESULTATS, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Appareil :', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '(pas de GPU : ce sera lent)')

HP = dict(dim=64, hidden=128, couches_graphe=2, dropout=0.2, lr=1e-3, batch=64, epoques_max=60, patience=5)
GRAINE = 42

In [ ]:
# Cellule 2 — Séquences, graphe expert, et repérage des PREMIÈRES tentatives
exos = pd.read_csv(os.path.join(PREP, 'exercices.csv'))
N = len(exos)
A_expert = np.load(os.path.join(PREP, 'adjacence_expert.npy')).astype(np.float32)
aretes_expert = list(zip(*np.nonzero(A_expert)))

def charger(partie):
    df = pd.read_csv(os.path.join(PREP, f'sequences_{partie}.csv')).sort_values(['uid', 'morceau'])
    U, Q, R, F = [], [], [], []
    vus = {}
    for uid, ex, rep in zip(df['uid'], df['exercices'], df['reponses']):
        q = np.array(ex.split(','), dtype=np.int64)
        r = np.array(rep.split(','), dtype=np.int64)
        deja = vus.setdefault(uid, set())          # exercices déjà tentés par cet élève (tous morceaux confondus)
        f = np.zeros(len(q), dtype=bool)
        for i, x in enumerate(q):
            if x not in deja:
                f[i] = True
                deja.add(x)
        U.append(uid); Q.append(q); R.append(r); F.append(f)
    return np.array(U), Q, R, F

donnees = {p: charger(p) for p in ['train', 'valid', 'test']}
for p, (u, q, r, f) in donnees.items():
    n_rep = sum(map(len, q)); n_first = sum(int(x.sum()) for x in f)
    print(f'{p:5s} : {len(set(u)):6,} élèves | {n_rep:9,} réponses | premières tentatives : {n_first:8,} ({n_first/n_rep:.1%})')

In [ ]:
# Cellule 3 — Tenseurs (Q exercices, R réponses, M masque, F première tentative)
def tenseurs(q_list, r_list, f_list):
    L = max(map(len, q_list))
    Q = np.zeros((len(q_list), L), dtype=np.int64); R = np.zeros_like(Q)
    M = np.zeros((len(q_list), L), dtype=bool);     F = np.zeros_like(M)
    for i, (q, r, f) in enumerate(zip(q_list, r_list, f_list)):
        Q[i, :len(q)], R[i, :len(r)], M[i, :len(q)], F[i, :len(f)] = q, r, True, f
    return tuple(torch.from_numpy(x) for x in (Q, R, M, F))

def sous_ensemble_train(fraction, graine=GRAINE):
    u, q, r, f = donnees['train']
    if fraction >= 1:
        return q, r, f
    rng = np.random.default_rng(graine)
    eleves = np.unique(u)
    gardes = set(rng.choice(eleves, size=max(1, int(fraction * len(eleves))), replace=False))
    idx = [i for i, x in enumerate(u) if x in gardes]
    return [q[i] for i in idx], [r[i] for i in idx], [f[i] for i in idx]

T_valid = tenseurs(*donnees['valid'][1:])
T_test  = tenseurs(*donnees['test'][1:])
print('OK')

### Les modèles

- **GraphKT** : identique à l'étape 5 (le graphe lisse les représentations des exercices, avec un canal « prérequis » et un canal « successeurs »).
- **PrereqKT** : GraphKT **plus** 6 indicateurs calculés à chaque instant pour l'exercice à prédire :
  taux de réussite moyen de l'élève sur ses **prérequis** déjà tentés, part des prérequis déjà tentés, présence de prérequis, et les trois mêmes indicateurs pour ses **successeurs**.
  C'est la traduction directe de l'hypothèse « *s'il a échoué sur A, il échouera plus souvent sur B* ».
  Avec un graphe vide, ces indicateurs valent 0 : PrereqKT redevient GraphKT sans graphe.

In [ ]:
# Cellule 4 — Modèles
def normaliser_lignes(A):
    deg = A.sum(1, keepdims=True)
    return np.divide(A, deg, out=np.zeros_like(A), where=deg > 0)

class GraphKT(nn.Module):
    def __init__(self, n, hp, A, indicateurs=False):
        super().__init__()
        d, h = hp['dim'], hp['hidden']
        A = np.asarray(A, dtype=np.float32)
        self.register_buffer('P_pre', torch.from_numpy(normaliser_lignes(A.T.copy())))  # ligne j : prérequis de j
        self.register_buffer('P_suc', torch.from_numpy(normaliser_lignes(A.copy())))    # ligne i : successeurs de i
        self.n, self.indicateurs = n, indicateurs
        self.E = nn.Embedding(n, d)
        self.couches = nn.ModuleList([nn.ModuleDict({'pre': nn.Linear(d, d), 'suc': nn.Linear(d, d)})
                                      for _ in range(hp['couches_graphe'])])
        self.R = nn.Embedding(2, d)
        self.lstm = nn.LSTM(2 * d, h, batch_first=True)
        self.drop = nn.Dropout(hp['dropout'])
        n_ind = 6 if indicateurs else 0
        self.pred = nn.Sequential(nn.Linear(h + d + n_ind, h), nn.ReLU(), nn.Linear(h, 1))

    def representations(self):
        H = self.E.weight
        a_pre = (self.P_pre.sum(1, keepdim=True) > 0).float()
        a_suc = (self.P_suc.sum(1, keepdim=True) > 0).float()
        for c in self.couches:
            H = H + torch.relu(c['pre'](self.P_pre @ H)) * a_pre + torch.relu(c['suc'](self.P_suc @ H)) * a_suc
        return H

    def calcul_indicateurs(self, q, r):
        # Historique cumulé par exercice jusqu'à l'instant t : nombre d'essais et de réussites (B, L-1, N)
        B, L = q.shape
        essais = torch.zeros(B, L - 1, self.n, device=q.device)
        essais.scatter_(2, q[:, :-1].unsqueeze(-1), 1.0)
        reussites = essais * r[:, :-1].unsqueeze(-1).float()
        essais, reussites = essais.cumsum(1), reussites.cumsum(1)
        tente = (essais > 0).float()
        taux = reussites / essais.clamp(min=1)
        sortie = []
        for P in (self.P_pre, self.P_suc):
            voisins = (P[q[:, 1:]] > 0).float()                 # voisins de l'exercice à prédire (B, L-1, N)
            n_vois = voisins.sum(-1)
            n_tentes = (voisins * tente).sum(-1)
            moy = (voisins * tente * taux).sum(-1) / n_tentes.clamp(min=1)
            sortie += [moy, n_tentes / n_vois.clamp(min=1), (n_vois > 0).float()]
        return torch.stack(sortie, -1)

    def forward(self, q, r):
        H = self.representations()
        h, _ = self.lstm(torch.cat([H[q[:, :-1]], self.R(r[:, :-1])], dim=-1))
        x = [self.drop(h), H[q[:, 1:]]]
        if self.indicateurs:
            x.append(self.calcul_indicateurs(q, r))
        return self.pred(torch.cat(x, dim=-1)).squeeze(-1)

print('OK')

In [ ]:
# Cellule 5 — Graphes : expert, aucun, aléatoire (même nombre de flèches)
def vers_matrice(aretes, n=N):
    A = np.zeros((n, n), dtype=np.float32)
    for a, b in aretes:
        A[a, b] = 1
    return A

def graphe(nom, graine=GRAINE):
    if nom == 'expert':
        return A_expert
    if nom == 'aucun':
        return np.zeros((N, N), dtype=np.float32)
    rng = random.Random(graine)
    E = set()
    while len(E) < len(aretes_expert):
        a, b = rng.sample(range(N), 2)
        E.add((a, b))
    return vers_matrice(E)
print({g: int(graphe(g).sum()) for g in ['expert', 'aucun', 'aleatoire']})

In [ ]:
# Cellule 6 — Entraînement et évaluation par sous-ensemble
def fixer_graine(g):
    random.seed(g); np.random.seed(g); torch.manual_seed(g); torch.cuda.manual_seed_all(g)

@torch.no_grad()
def evaluer(modele, T, batch=128):
    modele.eval()
    Q, R, M, F = T
    ys, ps, fs = [], [], []
    for i in range(0, len(Q), batch):
        q, r, m, f = (x[i:i+batch].to(DEVICE) for x in (Q, R, M, F))
        p = torch.sigmoid(modele(q, r))
        c = m[:, 1:]
        ys.append(r[:, 1:][c].cpu().numpy()); ps.append(p[c].float().cpu().numpy()); fs.append(f[:, 1:][c].cpu().numpy())
    y, p, f = (np.concatenate(x) for x in (ys, ps, fs))
    auc = lambda s: round(float(roc_auc_score(y[s], p[s])), 4) if len(set(y[s])) == 2 else None
    tout = np.ones_like(f, dtype=bool)
    return {'AUC': auc(tout), 'AUC_1re': auc(f), 'AUC_rep': auc(~f), 'ACC': round(float(accuracy_score(y, p >= 0.5)), 4)}

def entrainer(nom_modele, A, fraction=1.0, graine=GRAINE, hp=HP):
    fixer_graine(graine)
    Q, R, M, F = tenseurs(*sous_ensemble_train(fraction, graine))
    modele = GraphKT(N, hp, A, indicateurs=(nom_modele == 'PrereqKT')).to(DEVICE)
    opt = torch.optim.Adam(modele.parameters(), lr=hp['lr'])
    perte = nn.BCEWithLogitsLoss()
    meilleur, etat, attente, t0 = -1, None, 0, time.time()
    for ep in range(hp['epoques_max']):
        modele.train()
        ordre = torch.randperm(len(Q))
        for i in range(0, len(Q), hp['batch']):
            idx = ordre[i:i + hp['batch']]
            q, r, m = Q[idx].to(DEVICE), R[idx].to(DEVICE), M[idx].to(DEVICE)
            c = m[:, 1:]
            l = perte(modele(q, r)[c], r[:, 1:][c].float())
            opt.zero_grad(); l.backward()
            nn.utils.clip_grad_norm_(modele.parameters(), 5.0)
            opt.step()
        v = evaluer(modele, T_valid)['AUC']
        print(f'   époque {ep+1:2d} | AUC valid {v:.4f} | {(time.time()-t0)/60:.1f} min')
        if v > meilleur + 1e-4:
            meilleur, etat, attente = v, copy.deepcopy(modele.state_dict()), 0
        else:
            attente += 1
            if attente >= hp['patience']:
                break
    modele.load_state_dict(etat)
    return {'AUC_valid': round(meilleur, 4), **{f'{k}_test': v for k, v in evaluer(modele, T_test).items()},
            'epoques': ep + 1, 'minutes': round((time.time() - t0) / 60, 1)}
print('OK')

In [ ]:
# Cellule 7 — Expériences : 3 régimes × 5 graines × 2 modèles × 3 graphes (reprise automatique)
FRACTIONS = [0.01, 0.05, 0.10]
GRAINES   = [0, 1, 2, 3, 4]
MODELES   = ['PrereqKT', 'GraphKT']
GRAPHES   = ['expert', 'aleatoire', 'aucun']

chemin_res = os.path.join(RESULTATS, 'phase2_confirmation.csv')
resultats = pd.read_csv(chemin_res).to_dict('records') if os.path.exists(chemin_res) else []
fait = {(x['modele'], x['graphe'], float(x['fraction']), int(x['graine'])) for x in resultats}
total = len(FRACTIONS) * len(GRAINES) * len(MODELES) * len(GRAPHES)
t_debut = time.time()

for frac in FRACTIONS:
    for g_ in GRAINES:
        for nom in MODELES:
            for g in GRAPHES:
                if (nom, g, float(frac), g_) in fait:
                    continue
                print(f'\n>>> [{len(resultats)+1}/{total}] {nom} | {g} | {frac:.0%} | graine {g_}')
                res = entrainer(nom, graphe(g, graine=g_), fraction=frac, graine=g_)
                resultats.append({'modele': nom, 'graphe': g, 'fraction': frac, 'graine': g_, **res})
                pd.DataFrame(resultats).to_csv(chemin_res, index=False)
                print(f'   -> AUC 1res tentatives {res["AUC_1re_test"]} | tout {res["AUC_test"]} | '
                      f'{res["epoques"]} époques | total écoulé {(time.time()-t_debut)/60:.0f} min')

In [ ]:
# Cellule 8 — Résumé statistique à copier dans la conversation
tab = pd.DataFrame(resultats)
tab['plafond_atteint'] = tab['epoques'] >= HP['epoques_max']

print('=== Moyenne ± écart-type sur les graines ===')
agg = (tab.groupby(['fraction', 'modele', 'graphe'])
          .agg(n=('graine', 'nunique'),
               AUC_1re=('AUC_1re_test', 'mean'), sd_1re=('AUC_1re_test', 'std'),
               AUC_tout=('AUC_test', 'mean'), sd_tout=('AUC_test', 'std'),
               epoques=('epoques', 'mean'), plafond=('plafond_atteint', 'sum'))
          .round(4))
print(agg.to_string())

print('\n=== Écarts appariés (même graine = mêmes élèves) ===')
lignes = []
for (frac, nom), s in tab.groupby(['fraction', 'modele']):
    piv = s.pivot_table(index='graine', columns='graphe', values=['AUC_1re_test', 'AUC_test'])
    for col, lib in [('AUC_1re_test', '1res tentatives'), ('AUC_test', 'tout')]:
        for ref in ['aleatoire', 'aucun']:
            if (col, 'expert') not in piv or (col, ref) not in piv:
                continue
            d = (piv[(col, 'expert')] - piv[(col, ref)]).dropna()
            if len(d) < 2:
                continue
            t_p = stats.ttest_rel(piv[(col, 'expert')].loc[d.index], piv[(col, ref)].loc[d.index]).pvalue
            w_p = stats.wilcoxon(d, alternative='greater').pvalue if (d != 0).any() else 1.0
            lignes.append({'fraction': frac, 'modele': nom, 'mesure': lib, 'comparaison': f'expert − {ref}',
                           'n_graines': len(d), 'ecart_moyen': round(d.mean(), 4), 'ecart_sd': round(d.std(), 4),
                           'graines_gagnees': f'{int((d > 0).sum())}/{len(d)}',
                           'p_ttest_apparie': round(t_p, 4), 'p_wilcoxon_unilat': round(w_p, 4)})
ecarts = pd.DataFrame(lignes)
ecarts.to_csv(os.path.join(RESULTATS, 'phase2_confirmation_ecarts.csv'), index=False)
print(ecarts.to_string(index=False))
print("\nNote : avec 5 graines, la plus petite p-valeur possible du test de Wilcoxon unilatéral est 0,031 (5 graines gagnées sur 5).")